# 19 — Reward, return, and the discount factor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/019-rl-frame.ipynb)

A reinforcement-learning agent sees one number per step — a reward — and its real job is the total it collects over time, not the number in front of it. This notebook builds that total, called the return, and shows what the discount factor gamma does to it. You finish by dragging gamma and watching a patient plan turn into an impatient one.

## 1. Ten unit rewards

Here is the whole idea in one line of arithmetic. An episode gives ten rewards of 1, and the return is a weighted sum where step t counts for gamma to the power t. With gamma below 1 the far steps are worth less, so the total lands below 10.

In [ ]:
import numpy as np                              # the array library, preinstalled on Colab
gamma = 0.9                                     # how much a reward one step later is worth today
rewards = np.ones(10)                           # ten rewards, one per step, each worth 1
undiscounted = rewards.sum()                    # plain sum: every step counts the same
weights = gamma ** np.arange(10)                # gamma^0, gamma^1, ... gamma^9: the discount per step
discounted = (weights * rewards).sum()          # weight each reward, then add them up
print("rewards:", rewards)                      # ten 1.0s
print("weights:", np.round(weights, 3))         # 1.0, 0.9, 0.81, ...
print("undiscounted return:", undiscounted)     # 10.0
print("discounted return:", round(discounted, 3))   # 6.513, not 10

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, the array library Colab already has.</li><li><code>gamma = 0.9</code> — the discount factor: the exchange rate between now and one step later.</li><li><code>rewards = np.ones(10)</code> — the episode: ten steps, each paying 1.</li><li><code>undiscounted = rewards.sum()</code> — the naive total, every step equal.</li><li><code>weights = gamma ** np.arange(10)</code> — one weight per step, shrinking geometrically.</li><li><code>discounted = (weights * rewards).sum()</code> — multiply elementwise, then sum.</li><li><code>print(...)</code> — compare the two totals side by side.</li></ul>

</details>

## 2. Step by step: adding the return up

The vector formula hides the order of operations. Walk the episode instead: each step contributes its reward times its own weight, and the running total climbs fast at first and then flattens out. That flattening is the discount at work.

In [ ]:
total = 0.0                                    # the return collected so far
for t in range(10):                             # visit step 0, 1, ... 9
    weight = gamma ** t                         # this step's exchange rate
    contribution = weight * rewards[t]          # what this step adds to the return
    total = total + contribution                # fold it into the running total
    print(f"t={t}  weight={weight:.3f}  add={contribution:.3f}  total={total:.3f}")   # watch it flatten

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>total = 0.0</code> — the accumulator for the return.</li><li><code>for t in range(10)</code> — one iteration per step of the episode.</li><li><code>weight = gamma ** t</code> — the discount applied to step <code>t</code>.</li><li><code>contribution = ...</code> — reward at <code>t</code> times its weight.</li><li><code>total = total + contribution</code> — the new running return.</li><li><code>print(f"...")</code> — an f-string; <code>:.3f</code> prints three decimals.</li></ul>

</details>

## 3. Plot: the return as gamma moves

Turn the discount factor from 0 to almost 1 and recompute the same ten rewards. The curve rises from 1 to 10 and bends over as it gets close to 1, where the weights stop shrinking at all.

In [ ]:
import matplotlib.pyplot as plt                # plotting library, preinstalled on Colab

gammas = np.linspace(0.0, 0.99, 100)            # 100 candidate discount factors
returns = []                                    # one return per candidate gamma
for g in gammas:                                # recompute the episode's return each time
    returns.append((g ** np.arange(10)).sum())  # sum of g^0 .. g^9

plt.figure(figsize=(6, 3))                      # a wide, short figure
plt.plot(gammas, returns, color="tab:blue")     # return against gamma
plt.axvline(0.9, color="grey", linestyle="--")  # the gamma used in section 1
plt.axhline(10, color="grey", linestyle=":")    # the ceiling: gamma = 1
plt.xlabel("discount factor gamma")             # x-axis label
plt.ylabel("return of ten unit rewards")        # y-axis label
plt.title("gamma near 1 keeps the far steps; gamma near 0 throws them away")   # the takeaway
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — the plotting library, aliased to <code>plt</code>.</li><li><code>np.linspace(0.0, 0.99, 100)</code> — 100 evenly spaced gammas, skipping exactly 1.</li><li><code>returns = []</code> — an empty list to collect one number per gamma.</li><li><code>for g in gammas</code> — recompute the whole episode for each gamma.</li><li><code>returns.append(...)</code> — the sum of the ten discounted rewards.</li><li><code>plt.plot(...)</code> — the curve; the calls below only label and annotate it.</li><li><code>plt.axvline(0.9)</code> — marks the gamma from section 1 for comparison.</li></ul>

</details>

## 4. Interactive: drag gamma

Each bar is one step's contribution to the return, and the title reports the total. Drag the slider to the right and the tail of the bars lifts; drag it to zero and only the first bar survives.

In [ ]:
from ipywidgets import interact               # sliders that rerun a function

def show(gamma=0.9):                            # gamma comes from the slider
    steps = np.arange(10)                       # step indices 0 .. 9
    contrib = gamma ** steps                    # value of each step's unit reward
    total = contrib.sum()                       # the return for this gamma
    plt.figure(figsize=(6, 3))                  # a fresh figure for every drag
    plt.bar(steps, contrib, color="tab:blue")   # one bar per step
    plt.ylim(0, 1.05)                           # fixed axis, so bars do not jump
    plt.xlabel("step t")                        # x-axis label
    plt.ylabel("what step t's reward is worth")  # y-axis label
    plt.title(f"gamma={gamma:.2f} -> return {total:.2f} of a possible 10")   # live readout
    plt.show()                                  # draw it

interact(show, gamma=(0.0, 0.99, 0.01))         # the slider: range and step

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def show(gamma=0.9)</code> — the function that redraws one frame.</li><li><code>steps = np.arange(10)</code> — the step numbers, used as bar positions.</li><li><code>contrib = gamma ** steps</code> — each step's unit reward, discounted.</li><li><code>total = contrib.sum()</code> — the return shown in the title.</li><li><code>plt.bar(...)</code> — one bar per step, so the shape of the discount is visible.</li><li><code>plt.ylim(0, 1.05)</code> — pinning the y-axis keeps the drag readable.</li><li><code>interact(show, gamma=(0.0, 0.99, 0.01))</code> — attach the slider and its step size.</li></ul>

</details>

## 5. When it breaks: gamma = 0 is myopia

Two plans compete: 10 right now, or 100 ten steps from now. At gamma = 0 the second plan is worth exactly nothing, so the agent takes the 10 and walks away from a tenfold payoff. The crossing point near 0.79 is the whole cost of impatience.

In [ ]:
gammas = np.linspace(0.0, 0.99, 100)            # sweep the discount factor
now = np.full_like(gammas, 10.0)                # plan A: 10 at step 0, nothing after
later = 100 * gammas ** 10                      # plan B: nothing until step 10, then 100
lead = np.where(later > now, later, now)        # the value an optimal agent would report

plt.figure(figsize=(6, 3))                      # the comparison figure
plt.plot(gammas, now, color="tab:orange", label="10 now")            # plan A
plt.plot(gammas, later, color="tab:green", label="100 at step 10")   # plan B
plt.plot(gammas, lead, color="black", linestyle=":", label="best of the two")   # the upper envelope
plt.axvline(0.79, color="grey", linestyle="--")  # where the two plans tie
plt.xlabel("discount factor gamma")             # x-axis label
plt.ylabel("value of the plan")                 # y-axis label
plt.legend(loc="upper left")                    # name the lines
plt.title("at gamma=0 the 10-now plan wins; by gamma=0.9 the patient plan is worth 34.9")   # takeaway
plt.show()                                      # render

print("gamma=0.0 ->", now[0], "vs", round(later[0], 2))              # 10.0 vs 0.0: pure myopia
print("gamma=0.9 ->", now[90], "vs", round(later[90], 2))            # 10.0 vs 34.9: patience pays

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>np.full_like(gammas, 10.0)</code> — plan A's value, the same constant for every gamma.</li><li><code>later = 100 * gammas ** 10</code> — plan B is ten steps away, so it is scaled by gamma to the tenth.</li><li><code>np.where(later &gt; now, later, now)</code> — picks the better plan at each gamma.</li><li><code>plt.axvline(0.79)</code> — the gamma where the two plans are worth the same.</li><li><code>print(...)</code> — the two ends of the range, where the choice flips.</li></ul>

</details>

## Your turn

Before rerunning section 5, predict the value of the patient plan at gamma = 0.5. Then change the delay from ten steps to twenty and watch the crossing point move right — how much patience does a longer wait demand?